# Reading Giulia Parquet outputs — `OutputReader` examples

After a run, Giulia writes its results as Parquet **datasets** in the output directory
(`output.directory` in `config.yml`, e.g. `outputs/gof`). Each Saveable *label* becomes one
dataset named `<project>_<label>` (e.g. `..._ue_kpi`, `..._cell_kpi`).

This notebook shows the [`OutputReader`](../giulia/outputs/read_outputs.py) API, which answers
the four things you usually want:

1. **how many** datasets there are, and which — `count()` / `datasets()`
2. the **schema** (columns + types) — `schema()` / `kpis()`
3. a **whole KPI column** — `column(dataset, kpi)`
4. a KPI for **one episode** (and optionally one step) — `values(dataset, kpi, episode, step)`

> Run the cells with the **`Giulia_gpu`** kernel (the `python3` kernel of that conda env).
> Full guide: `docs/output-saveable/reading-parquet.md`.

## Setup

Make the repo importable and run from its root so paths like `outputs/gof` resolve.

In [1]:
import os
import sys
import glob

# Walk up to the repository root (the folder that contains 'giulia/').
root = os.getcwd()
while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, "giulia")):
    root = os.path.dirname(root)
os.chdir(root)
sys.path.insert(0, root)
print("Repo root:", root)

Repo root: C:\Users\Matteo\UPV_Matteo\Working_Directory\Giulia


In [2]:
from giulia.outputs import OutputReader

# Point this at your run's output folder (output.directory in config.yml).
OUTPUT_DIR = "outputs/gof"

# Fallback for this example: if that folder is missing, use the first outputs/ folder
# that contains Parquet files.
if not os.path.isdir(OUTPUT_DIR):
    found = sorted(glob.glob("outputs/**/*.parquet", recursive=True))
    if found:
        OUTPUT_DIR = os.path.dirname(found[0])

reader = OutputReader(OUTPUT_DIR)
print("Reading from:", OUTPUT_DIR)

Reading from: outputs/gof


## 1. How many datasets are there, and which?

In [3]:
print("Number of datasets:", reader.count())
reader.datasets()

Number of datasets: 6


['3GPPTR38_901_5G_uniform_bbu_kpi',
 '3GPPTR38_901_5G_uniform_cell_kpi',
 '3GPPTR38_901_5G_uniform_csiRsBeam_kpi',
 '3GPPTR38_901_5G_uniform_ru_kpi',
 '3GPPTR38_901_5G_uniform_ssbBeam_kpi',
 '3GPPTR38_901_5G_uniform_ue_kpi']

## 2. The schema — which columns and which types

`schema()` reads the column names and types straight from the file **metadata** (no row data is
loaded). A dataset is addressed by its **full name** or any **unique suffix** (e.g. `"ue_kpi"`).

In [4]:
# Choose a dataset to explore by NAME (a short suffix like "cell_kpi" also works).
dataset = "3GPPTR38_901_5G_uniform_cell_kpi"
print("Exploring:", dataset)

schema = reader.schema(dataset)        # {column: type}
print("Total columns:", len(schema))
dict(list(schema.items())[:10])        # first 10 columns

Exploring: 3GPPTR38_901_5G_uniform_cell_kpi
Total columns: 5


{'episode_idx': 'uint64',
 'step_idx': 'uint64',
 'cell_id': 'uint64',
 'Cell_Selection_SSB.ues_per_cell': 'int32',
 'BaseStations.cell_radio_unit_statuses': 'string'}

In [5]:
# The KPI columns only (the index columns episode_idx / step_idx / *_id are dropped):
kpis = reader.kpis(dataset)
print("KPI columns:", len(kpis))
kpis[:10]

KPI columns: 2


['Cell_Selection_SSB.ues_per_cell', 'BaseStations.cell_radio_unit_statuses']

## 3. A whole KPI column

`column(dataset, kpi)` returns the entire column as a pandas `Series` (across every episode,
step and entity). Only that one column is read from disk.

In [6]:
# Choose a KPI by NAME (use the reader.kpis(dataset) list above to see the options):
kpi = "Cell_Selection_SSB.ues_per_cell"
column = reader.column(dataset, kpi)
print("KPI:", kpi, "| length:", len(column))
column.head()

KPI: Cell_Selection_SSB.ues_per_cell | length: 513


0     7
1    11
2    12
3     7
4    13
Name: Cell_Selection_SSB.ues_per_cell, dtype: int32

## 4. A KPI for one episode (and optionally one step)

`values(...)` filters by `episode` (and `step`) using Parquet row filters, so non-matching rows
are never read. **The shape depends on the file:**

- **exploded** file (`ue_kpi`, `cell_kpi`, …) → one value per entity (one row per `*_id`).
- **generic** file → one row holds the whole per-entity vector; use `.iloc[0]` to get that list.

In [7]:
values = reader.values(dataset, kpi, episode=0, step=0)
print(f"{kpi} @ episode 0, step 0  ({len(values)} values)")
values.tolist()[:20]

Cell_Selection_SSB.ues_per_cell @ episode 0, step 0  (57 values)


[7, 11, 12, 7, 13, 7, 8, 14, 6, 10, 13, 14, 8, 13, 13, 11, 10, 12, 10, 6]

## Extras

In [8]:
# A unique suffix resolves to the same dataset as the full name:
reader.schema("cell_kpi") == reader.schema(dataset)

True

In [9]:
# The whole table as a DataFrame (optionally just a few columns):
df = reader.read(dataset, columns=["episode_idx", "step_idx", kpi])
df.head()

,episode_idx,step_idx,Cell_Selection_SSB.ues_per_cell
0,0,0,7
1,0,0,11
2,0,0,12
3,0,0,7
4,0,0,13


In [10]:
# Generic vs exploded: if a 'generic' dataset exists, one row holds the whole vector.
generic = [d for d in reader.datasets() if d.endswith("generic")]
if generic:
    g = generic[0]
    gkpi = reader.kpis(g)[0]
    row = reader.values(g, gkpi, episode=0, step=0)
    print("generic file: one row whose cell is the whole vector ->")
    print(row.iloc[0] if len(row) else "no data for episode 0, step 0")
else:
    print("No 'generic' dataset here — all datasets are exploded (one row per entity).")

No 'generic' dataset here — all datasets are exploded (one row per entity).


In [11]:
# A wrong KPI (or dataset) name raises a clear KeyError listing what is available:
try:
    reader.column(dataset, "Nope.not_a_kpi")
except KeyError as error:
    print("KeyError:", error)

KeyError: "KPI 'Nope.not_a_kpi' is not in dataset '3GPPTR38_901_5G_uniform_cell_kpi'. It has 2 KPIs; e.g. ['Cell_Selection_SSB.ues_per_cell', 'BaseStations.cell_radio_unit_statuses']"


## Tips

- Address a dataset by its **full name** or a **unique suffix**.
- `schema()` reads only metadata — fast, no data load.
- `column()` / `values()` read just the one column thanks to Parquet **column pruning**.
- From a terminal you can also run: `python -m giulia.outputs.read_outputs outputs/gof`.
- See `docs/output-saveable/reading-parquet.md` for the full guide and the raw-pandas equivalent.